# Plantilla de variante

**Autor:** _tu nombre_ · **Variante:** _nombre corto_

Cómo usarla:
1. Cópiala como `notebooks/NN_autor_tema.ipynb` (no edites esta).
2. Sustituye `fit_predict` por tu variante. Todo lo que use etiquetas se ajusta **solo** con `train_df` (ver `AGENTS.md`).
3. Ejecuta de arriba abajo, apunta la fila que imprime la evaluación en `EXPERIMENTOS.md` y guarda el notebook con salidas.

Aquí `fit_predict` predice la media de train: es la referencia mínima (RMSE ≈ 30,4) y sirve para comprobar que todo funciona.

## Preparación (local o Colab)

In [1]:
import os, sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["TECLIN_DATA"] = "/content/drive/MyDrive/TecLin/data"  # <- carpeta con raw/ y processed/
    RAMA = "main"                                                      # <- tu rama
    REPO = Path("/content/TecLin_Final_Boss")
    if not REPO.exists():
        !git clone -q -b {RAMA} https://github.com/Vija97/TecLin_Final_Boss.git {REPO}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))

In [2]:
import numpy as np
import pandas as pd

from src.config import TRAITS
from src.data import cargar_autores, textos_por_usuario
from src.evaluate import PorRasgo, validacion_cruzada
from src.submission import generar_entrega

train, test = cargar_autores()
print(f"train: {len(train)} usuarios · test: {len(test)} usuarios")

train: 1288 usuarios · test: 323 usuarios


## Datos de texto

`textos_por_usuario()` devuelve el texto concatenado de cada usuario (train + test; los que no tienen posts reciben `""`).
Con `max_posts=N` se limita a N posts por usuario (muestra reproducible): útil para probar rápido.

In [3]:
# textos = textos_por_usuario(max_posts=200)   # descomenta si tu variante usa texto

## Variante

`fit_predict(train_df, val_df)` recibe filas de `authors_train` y devuelve un array `(len(val_df), 5)` en el orden de `TRAITS`.

In [4]:
from sklearn.dummy import DummyRegressor


def fit_predict(train_df: pd.DataFrame, val_df: pd.DataFrame) -> np.ndarray:
    # Sustituir por la variante real. PorRasgo ignora las etiquetas NaN de cada rasgo.
    X_tr, X_va = np.zeros((len(train_df), 1)), np.zeros((len(val_df), 1))
    modelo = PorRasgo(DummyRegressor(strategy="mean")).fit(X_tr, train_df[TRAITS].to_numpy())
    return modelo.predict(X_va)

## Evaluación (validación cruzada común: no cambiar)

In [5]:
res = validacion_cruzada(fit_predict, train, nombre="media")

media: RMSE medio 30.406 (folds [30.698, 30.617, 29.821, 30.573, 30.286])
| agreeableness | openness | conscientiousness | extraversion | neuroticism | medio |
| 31.23 | 27.85 | 30.03 | 30.62 | 32.30 | **30.41** |


## Resultados y conclusión

_Qué se probó, qué RMSE salió frente a la mejor variante y qué se aprende. Esto se copia luego al notebook final._

## Generar el CSV de entrega (solo si esta variante se va a enviar)

Entrena con todo train y predice test. Ver la skill `entrega`.

In [6]:
# preds_test = fit_predict(train, test)
# generar_entrega(test, preds_test, "media")